# Transformer 从零学习：PyTorch + Google Colab

[返回项目首页](https://github.com/Maple-Ze/Transformer-to-VLA) · [配套笔记](https://github.com/Maple-Ze/Transformer-to-VLA/blob/main/01_Attention_Fundamentals/01_Attention_Fundamentals.md)

**目标**：先理解张量运算与 Self-Attention，再亲手写 Multi-Head Attention、位置编码、因果掩码、Transformer Block，最后训练一个微型自回归语言模型（Mini-GPT）。

**运行方法**：在 Google Colab 中依次执行每一个代码单元格。CPU 即可运行；启用 GPU 会更快。不调用 `nn.Transformer` 或 `nn.MultiheadAttention`，核心机制自己实现。

**学习顺序**：① Q/K/V → ② 注意力矩阵 → ③ 多头 → ④ mask → ⑤ 位置 → ⑥ Transformer Block → ⑦ 训练 → ⑧ 自回归生成。

符号：`B` 批大小、`T` 序列长度、`C` 隐藏维度、`H` 头数。

In [ ]:
import math, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

seed = 42
random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch:', torch.__version__, '| device:', device)

## 1. 从一个矩阵乘法开始：Scaled Dot-Product Attention

输入 `x: [B,T,C]`，先通过三个线性投影得到 `Q,K,V`。注意力计算：

$$A=\operatorname{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right),\qquad Y=AV.$$

**先不要训练模型**：观察不同 token 对彼此的注意力权重。

In [ ]:
B, T, C = 1, 5, 8
x = torch.randn(B, T, C)
Wq, Wk, Wv = (nn.Linear(C, C, bias=False) for _ in range(3))
Q, K, V = Wq(x), Wk(x), Wv(x)
scores = Q @ K.transpose(-2, -1) / math.sqrt(C)  # [1,5,5]
weights = F.softmax(scores, dim=-1)
y = weights @ V
print('Q:', Q.shape, 'scores:', scores.shape, 'output:', y.shape)
print('每行权重和：', weights.sum(-1))
assert torch.allclose(weights.sum(-1), torch.ones(B, T), atol=1e-6)
plt.figure(figsize=(5,4))
plt.imshow(weights[0].detach().numpy(), cmap='Blues', vmin=0, vmax=1)
plt.colorbar(label='attention weight')
plt.xlabel('key token'); plt.ylabel('query token'); plt.title('Self-attention (untrained)')
plt.show()

**思考**：为什么 `QKᵀ` 的形状是 `[T,T]`？为什么 softmax 沿最后一维？这里的权重来自随机参数，**不能解释为模型学到的语义关联**。

## 2. 因果掩码（Causal Mask）

训练下一 token 预测模型时，位置 `t` 不能偷看位置 `t+1` 及其后面的词。把未来位置的注意力分数设为 `-inf`，再做 softmax。

In [ ]:
causal = torch.tril(torch.ones(T, T, dtype=torch.bool))
masked_scores = scores.masked_fill(~causal, float('-inf'))
masked_weights = F.softmax(masked_scores, dim=-1)
print(masked_weights[0].detach().round(decimals=3))
assert torch.all(masked_weights[0].triu(diagonal=1) == 0)
plt.figure(figsize=(5,4))
plt.imshow(masked_weights[0].detach().numpy(), cmap='Blues', vmin=0, vmax=1)
plt.colorbar(); plt.title('Causal self-attention'); plt.xlabel('key token'); plt.ylabel('query token')
plt.show()

## 3. 手写 Multi-Head Self-Attention

每个头在不同子空间中计算注意力，再将结果拼接并线性映射回 `C` 维。形状：`[B,T,C] → [B,H,T,C/H] → [B,T,C]`。

In [ ]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.0):
        super().__init__()
        assert d_model % num_heads == 0
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.out = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, causal=True, return_attention=False):
        B, T, C = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        def split_heads(t):
            return t.reshape(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        q, k, v = map(split_heads, (q, k, v))  # [B,H,T,D]
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        if causal:
            mask = torch.tril(torch.ones(T, T, device=x.device, dtype=torch.bool))
            scores = scores.masked_fill(~mask, float('-inf'))
        attn = F.softmax(scores, dim=-1)
        context = self.dropout(attn) @ v
        context = context.transpose(1, 2).contiguous().reshape(B, T, C)
        result = self.out(context)
        if return_attention:
            return result, attn  # pre-dropout attention weights
        return result

mha = MultiHeadSelfAttention(d_model=32, num_heads=4)
test_x = torch.randn(2, 6, 32)
test_y, test_a = mha(test_x, return_attention=True)
print('output:', test_y.shape, '| attention:', test_a.shape)
assert test_y.shape == test_x.shape
assert test_a.shape == (2, 4, 6, 6)

## 4. 位置编码和 Transformer Block

仅靠 Attention 不知道 token 的顺序。这里用**可学习位置向量**（Mini-GPT 的常见方式），并使用 Pre-LN 残差结构：

`x ← x + Attention(LayerNorm(x))`  
`x ← x + MLP(LayerNorm(x))`

这是一种 **decoder-only、causal** 的 Transformer，和原始 encoder-decoder Transformer 不是同一个完整架构。

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadSelfAttention(d_model, num_heads, dropout)
        self.norm2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, 4*d_model), nn.GELU(),
            nn.Linear(4*d_model, d_model), nn.Dropout(dropout)
        )
    def forward(self, x):
        x = x + self.attn(self.norm1(x), causal=True)
        x = x + self.mlp(self.norm2(x))
        return x

block = TransformerBlock(d_model=32, num_heads=4)
print('block output:', block(torch.randn(2, 6, 32)).shape)

## 5. 完整 Mini-GPT：预测下一个 token

任务：给定 `0…9` 的循环序列，例如 `3,4,5,6`，预测下一个 token 是 `7`。这是**人为设计的入门 toy task**，目的是检查模型训练、遮罩与生成是否连贯，**不表示真实语言理解**。

输入 `idx: [B,T]`，模型返回 `logits: [B,T,vocab_size]`。

In [ ]:
class MiniGPT(nn.Module):
    def __init__(self, vocab_size=10, context_length=16, d_model=32, num_heads=4, num_layers=2):
        super().__init__()
        self.context_length = context_length
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.position_emb = nn.Embedding(context_length, d_model)
        self.blocks = nn.Sequential(*[
            TransformerBlock(d_model, num_heads) for _ in range(num_layers)
        ])
        self.norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size)
    def forward(self, idx, targets=None):
        B, T = idx.shape
        assert T <= self.context_length
        pos = torch.arange(T, device=idx.device)
        x = self.token_emb(idx) + self.position_emb(pos)[None, :, :]
        logits = self.lm_head(self.norm(self.blocks(x)))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss

model = MiniGPT().to(device)
print('trainable parameters:', sum(p.numel() for p in model.parameters() if p.requires_grad))
print('test logits:', model(torch.randint(0,10,(2,8),device=device))[0].shape)

## 6. 构建训练数据和优化循环

训练样本中，每个位置的监督标签都是**下一位置的数字**。随机起点避免模型只记住单一输入。CPU 环境可先训练 200 步；如果 loss 尚未明显降低可继续训练。

In [ ]:
def make_batch(batch_size=64, seq_len=12, vocab_size=10):
    starts = torch.randint(0, vocab_size, (batch_size,1), device=device)
    offsets = torch.arange(seq_len+1, device=device)[None,:]
    seq = (starts + offsets) % vocab_size
    return seq[:,:-1], seq[:,1:]

X, Y = make_batch(batch_size=2, seq_len=8)
print('X:', X.tolist())
print('Y:', Y.tolist())

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3)
losses = []
model.train()
for step in range(301):
    xb, yb = make_batch()
    _, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
    if step % 50 == 0:
        print(f'step {step:03d} | loss {loss.item():.4f}')
plt.figure(figsize=(6,3))
plt.plot(losses); plt.xlabel('training step'); plt.ylabel('cross-entropy loss'); plt.title('Mini-GPT training')
plt.grid(alpha=0.2); plt.show()

## 7. 自回归生成：一次只生成一个 token

给出起始 token，取最后位置的 logits 预测下一个 token，将预测值接到序列末尾，重复执行。为便于观察，使用 `argmax`（贪心解码）。

In [ ]:
@torch.no_grad()
def generate(model, prompt, num_new_tokens):
    model.eval()
    idx = torch.tensor([prompt], dtype=torch.long, device=device)
    for _ in range(num_new_tokens):
        context = idx[:, -model.context_length:]
        logits, _ = model(context)
        next_token = logits[:, -1, :].argmax(dim=-1, keepdim=True)
        idx = torch.cat((idx, next_token), dim=1)
    return idx[0].tolist()

for start in [0, 3, 7]:
    print('generated:', generate(model, [start], 15))

## 8. 验证因果性（非常重要）

修改未来 token，前面位置的输出不应变化。这是因果掩码是否正确的直接单元测试。

In [ ]:
model.eval()
a = torch.tensor([[1,2,3,4,5]], device=device)
b = torch.tensor([[1,2,3,9,9]], device=device)
with torch.no_grad():
    la, _ = model(a)
    lb, _ = model(b)
print('前 3 个位置 logits 的最大差异:', (la[:,:3] - lb[:,:3]).abs().max().item())
assert torch.allclose(la[:,:3], lb[:,:3], atol=1e-5)
print('causal test passed')

## 9. 练习（建议逐个完成）

1. **改头数**：把 `num_heads=4` 改为 `1`、`8`，检查张量形状与 loss。
2. **关掉 mask**：在注意力模块中设置 `causal=False`（需要修改 Block 调用），观察因果测试为何失败；注意训练损失低并不代表模型可以正确自回归生成。
3. **去掉位置编码**：将 `+ self.position_emb(pos)[None,:,:]` 删掉，比较效果（简单循环任务可能几乎不受影响，改用依赖顺序的任务更明显）。
4. **可视化训练后的头**：调用 `model.blocks[0].attn(..., return_attention=True)`；记得将嵌入输入经过 `norm1`，观察每个头的注意力矩阵。
5. **下一阶段**：Encoder-only 的序列分类、Encoder–Decoder 的序列翻转、真正的文本分词与文本 next-token 训练。

### 一个容易混淆的关键点

`Q、K、V` **不是三个输入句子**，而是同一输入的不同线性投影；Self-Attention 负责在序列位置间交换信息，MLP 负责逐位置的非线性变换，残差连接与归一化帮助深层训练。

---

[返回项目首页](https://github.com/Maple-Ze/Transformer-to-VLA) · [下一章：02 · Encoder-Decoder Transformer](https://github.com/Maple-Ze/Transformer-to-VLA/blob/main/02_Encoder_Decoder_Transformer/02_Encoder_Decoder_Transformer.ipynb)